## Preprocess Data

In [5]:
# %%writefile ../build/books_build.py
"""
books_build.py
----------------
Preprocess the GoodBooks-10k dataset and build FAISS index.
This script is meant to be run once to generate embeddings and save the index.
"""

import pandas as pd
from langchain_core.documents import Document
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS



def build_books_index():
    # --- PREPROCESS DATA ---
    books = pd.read_csv("../data/goodbooks-10k/books.csv")
    book_tags = pd.read_csv("../data/goodbooks-10k/book_tags.csv")
    tags = pd.read_csv("../data/goodbooks-10k/tags.csv")
    ratings_books = pd.read_csv("../data/goodbooks-10k/ratings.csv")
    to_read = pd.read_csv("../data/goodbooks-10k/to_read.csv")

    # Step 1: Join book_tags with tags → enrich with tag names
    book_tags_named = book_tags.merge(tags, on="tag_id")

    # Step 2: Merge with books using goodreads_book_id → attach metadata
    books_enriched = books.merge(book_tags_named, on="goodreads_book_id", how="left")

    # Step 3: Compute average rating per book_id → quality signal
    avg_ratings = ratings_books.groupby("book_id")["rating"].mean().reset_index()
    books_enriched = books_enriched.merge(avg_ratings, on="book_id", how="left")

    # Step 4: Count how many users marked each book as to-read → popularity signal
    to_read_count = to_read.groupby("book_id").size().reset_index(name="to_read_count")
    books_enriched = books_enriched.merge(to_read_count, on="book_id", how="left")

    # Step 5: Clean up NaN values for readability
    books_enriched["rating"] = books_enriched["rating"].fillna("No rating")
    books_enriched["tag_name"] = books_enriched["tag_name"].fillna("No tags")

    # Step 6: Group tags into a list per book
    books_grouped = books_enriched.groupby(
        ["book_id", "title", "authors", "original_publication_year", "rating", "to_read_count"]
    )["tag_name"].apply(list).reset_index()

    # Step 7: Convert DataFrame rows into LangChain Documents
    book_docs = []
    for _, row in books_grouped.iterrows():
        text = f"""
        Book: {row['title']}
        Authors: {row['authors']}
        Publication Year: {row['original_publication_year']}
        Rating: {row['rating']}
        Reading Count: {row['to_read_count']}
        Genres: {', '.join(row['tag_name'])}
        """
        book_docs.append(Document(page_content=text.strip()))

    print(f"✅ Prepared {len(book_docs)} book documents.")

    # Step 8: Embeddings → convert text chunks into dense vectors
    embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

    # Step 9: Vectorstore → store embeddings in FAISS for efficient similarity search
    vectorstore = FAISS.from_documents(book_docs, embeddings)

    # Step 10: Save FAISS index
    vectorstore.save_local("../data/faiss_books_index")
    print("✅ Books FAISS index built and saved at ../data/faiss_books_index")


# Only run build when executed directly, not on import
if __name__ == "__main__":
    build_books_index()


✅ Prepared 9965 book documents.
✅ Books FAISS index built and saved at ../data/faiss_books_index


## Validate Data

In [ ]:
# Total rows
total_rows = len(books_grouped)

# Count missing and non-missing per column
missing_count = books_grouped.isnull().sum()
non_missing_count = total_rows - missing_count
missing_percent = (missing_count / total_rows) * 100

# Combine into one DataFrame
missing_summary = pd.DataFrame({
    "non_missing": non_missing_count,
    "missing": missing_count,
    "missing_percent": missing_percent.round(2)
})

print(missing_summary)

## Load Variables

In [3]:
# load API key from .env 
from dotenv import load_dotenv, find_dotenv 
load_dotenv(find_dotenv())
# llm model
model="gpt-4o-mini"

## Business Logic

In [2]:
# %%writefile ../src/chain_books.py
"""
chain_books.py
----------------
Core LangChain module for BooksRAG.

Responsibilities:
- Load the persisted FAISS index built by books_build.py
- Define a retrieval-augmented generation (RAG) chain for book queries
- Designed for integration into the CultRAG pipeline
"""

# --- BUSINESS LOGIC ---

# Core LangChain modules
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate

# Embeddings + Vectorstore
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from utils.paths import DATA_DIR


# Step 1: Load persisted FAISS index (built once in build/books_build.py)
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
vectorstore = FAISS.load_local(
    str(DATA_DIR / "faiss_books_index"),
    embeddings,
    allow_dangerous_deserialization=True
)
retriever = vectorstore.as_retriever()

# Step 2: LLM → define the language model
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

# Step 3: Prompt Template → instructions for how the assistant should answer

prompt_books = ChatPromptTemplate.from_template("""
You are a book recommendation assistant using retrieval-augmented generation (RAG).

Use ONLY the provided context to answer the question.
If the answer is not in the context, say "Not found in the catalog."

Conversation so far:
{history}

Context:
{context}

Question:
{question}

Rules:
- ONLY include BOOKS in your answer.
- DO NOT include movies, songs, or any other media, even if present in the context.
- Ignore any non-book entries in the context completely.
- Do NOT add items that are not in the context.
- Do NOT guess or hallucinate.
- Do NOT mention other domains (movies, songs) or their absence.
- Output a valid markdown table with headers.
- Include a short summary after the table.

Answer:
""")


# Step 4: Helper to format retrieved docs into a single string
def format_docs(docs):
    """
    Convert a list of LangChain Document objects into a single string.
    Each document’s page_content is joined with double newlines.
    Used to feed retrieved context into the prompt.
    """
    return "\n\n".join([d.page_content for d in docs])


# Step 5: Build the Core LCEL Retrieval Chain
chain_books = (
    {
        "context": lambda x: retriever.invoke(x["question"]),
        "question": lambda x: x["question"],
        "history": lambda x: x.get("history", "")
    }
    | prompt_books
    | llm
)

Overwriting ../src/chain_books.py


## Query & Display

In [6]:
# import sys, os
# sys.path.append(os.path.abspath(".."))  # go up one level from notebooks/

# from src.chain_books import chain_books

# Display helpers for Jupyter
from IPython.display import display, Markdown

# --- 8. Query ---
query = "Please list 3 best fantasy books"

response = chain_books.invoke({"question": query})

# --- 9. Display ---
display(Markdown(response.content))

| Title                                                                 | Authors                     | Publication Year | Rating  | Reading Count |
|-----------------------------------------------------------------------|-----------------------------|------------------|---------|---------------|
| A Knight of the Seven Kingdoms (The Tales of Dunk and Egg, #1-3)    | George R.R. Martin, Gary Gianni | 2013             | 4.11    | 58            |
| Theft of Swords (The Riyria Revelations, #1-2)                       | Michael J. Sullivan         | 2011             | 4.09    | 175           |
| Wicked: The Life and Times of the Wicked Witch of the West (The Wicked Years, #1) | Gregory Maguire, Douglas Smith | 1995             | 3.37    | 866           |

These three books represent a mix of high fantasy and unique retellings within the genre, showcasing different styles and themes that appeal to fantasy readers.